**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Scaling Neural Networks

The [ANN](../Intro_ANN/Intro_ANN.ipynb) and [CNN](../Intro_CNN/Intro_CNN.ipynb) workshops trained models with thousands of parameters. Frontier models have *billions* — and the jump is not "the same but bigger": it changes what limits you (memory and data movement, not ideas), what you measure (throughput, utilization), and even what to expect (scaling laws). This workshop builds that systems mindset with experiments you can run on a laptop CPU.

> ℹ️ All benchmarks below run on **CPU** and demonstrate the *reasoning*; sections that only make sense on GPUs are clearly marked *illustrative — not executed here*.

## 0. Introduction

Three questions organize everything:

1. **Where do the parameters and FLOPs go?** (accounting)
2. **What limits my throughput?** (compute vs memory bandwidth — the [GPU workshop's](../../Intro_GPU/Intro_GPU.ipynb) CGMA ratio, at training scale)
3. **What does more compute buy?** (scaling laws)

## 1. Pre-requisites

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) and [Intro to ANN](../Intro_ANN/Intro_ANN.ipynb).
- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory-latency worldview.
- `pip install torch` (CPU build is fine here).

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
print(torch.__version__, "| threads:", torch.get_num_threads())

---
### 🕐 Session 1 of 2 — *Why Scale? Accounting & Scaling Laws* (~35 min)
**Goal:** count parameters and FLOPs; see diminishing-but-predictable returns in a toy scaling study.
**Builds on:** [ANN](../Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (making training fast).

---

## 2. Parameter & FLOP Accounting

💡 **Intuition.** Before optimizing anything, learn to *count*. A linear layer $d_{in} \to d_{out}$ stores $d_{in} d_{out} + d_{out}$ weights and spends $\approx 2 \, d_{in} d_{out}$ FLOPs per input (one multiply + one add per weight). Rule of thumb for training: **forward ≈ 2 FLOPs/param/token, backward ≈ twice the forward** — so training cost $\approx 6 \times$ params $\times$ tokens. That one line explains most headlines about GPU-months.

In [ ]:

# YOUR CODE HERE


## 3. A Toy Scaling Study

💡 **Intuition.** The famous scaling-law plots (loss vs compute, straight lines on log-log axes) are *empirical* — but you can reproduce their shape on a laptop. Fix a task, sweep model size with an equal training budget per size, and plot final loss vs parameters on log axes. Expect: big early gains, then a steady power-law-ish slide — and eventually a floor set by the data's intrinsic noise, which **no** amount of scale removes.

In [ ]:
# Task: regress y = sin(4x) + noise. The noise floor (var 0.01) is unbeatable BY DESIGN.

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


Real scaling laws sweep **data and compute** jointly (Chinchilla's lesson: params and tokens should grow together) — but the qualitative structure you just plotted is the same one governing billion-dollar training runs.

---
### 🕐 Session 2 of 2 — *Making Training Fast* (~40 min)
**Goal:** find the actual bottleneck: batch-size throughput curves, profiling, gradient accumulation, precision.
**Builds on:** Session 1.

---

## 4. Throughput vs Batch Size

💡 **Intuition.** Per-sample overhead (Python, kernel launches, optimizer bookkeeping) is *fixed*; compute grows with the batch. Small batches ⇒ overhead dominates and throughput climbs as batches grow; eventually arithmetic saturates the hardware and the curve flattens — or even *dips*, as very large batches spill out of cache (CPU) or run out of memory (GPU). **Measure the knee** — that's your efficient operating point.

In [ ]:

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


## 5. Profile Before You Optimize

Guessing at bottlenecks is how you optimize the wrong thing. PyTorch ships a profiler — read its table before touching your code.

In [ ]:

# YOUR CODE HERE


Reading it: `addmm`/`mm` rows are your matrix multiplies (real work); everything else is overhead. The ratio between them tells you whether to seek faster math or less overhead.

## 6. Gradient Accumulation

💡 **Intuition.** Want the optimization behavior of batch 1024 but only memory for 256? Run 4 micro-batches, **add up their gradients, step once**. Gradients are sums over samples, so the result is mathematically the large batch (identical when the loss averages per micro-batch of equal size and you scale by the count). This is the standard trick behind every 'effective batch size' line in a paper.

In [ ]:

# YOUR CODE HERE


## 7. The GPU-Scale Toolbox *(illustrative — not executed in this CPU notebook)*

On real accelerators the same reasoning continues with hardware-specific tools — presented here as a map, since none of this can be demonstrated honestly on CPU:

- **Mixed precision** (`torch.autocast` + fp16/bf16): tensor cores double-to-quadruple matmul throughput and halve activation memory; loss scaling guards small fp16 gradients.
- **Data parallelism** (`DistributedDataParallel`): replicate the model, split the batch, all-reduce gradients — gradient accumulation across machines, plus a network.
- **Memory arithmetic**: Adam training in fp32 costs ≈ 16 bytes/param (weights 4 + grads 4 + two moments 8) before activations — a 7B model wants ~112 GB, hence sharding (ZeRO/FSDP), activation checkpointing (recompute instead of store), and model/pipeline parallelism.

Each is the Session-2 mindset — *find the binding constraint, spend the cheap resource* — applied to a bigger machine.

## 8. Conclusion

Scaling is accounting plus bottleneck-hunting: count params and FLOPs, measure the throughput knee, profile before optimizing, accumulate when memory binds, and expect power-law returns onto a noise floor. The mindset transfers unchanged from laptop to cluster.

---
## Where next

- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory hierarchy all of this optimizes against.
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture the scaling laws were measured on.
- [Intro to OS](../../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — first-touch, scheduling, and why your warmup iterations exist.